## Module 7 — Derivatives Pricing and Greeks

In this module, I extended the platform from linear ETF positions to a nonlinear derivative product.

I priced a European call option on SPY using the Black-Scholes model and calculated the main Greeks:

- Delta measures sensitivity to the SPY price.
- Gamma measures how Delta changes when SPY price changes.
- Vega measures sensitivity to volatility.
- Theta measures time decay.
- Rho measures sensitivity to the risk-free interest rate.

I also performed simple sensitivity analysis by shocking the SPY price and volatility. This shows how option value changes under different market risk factor movements.


In [24]:
%%writefile /content/drive/MyDrive/market_risk_project/src/black_scholes.py

import importlib
import numpy as np
import pandas as pd
from scipy.stats import norm


class BlackScholesOption:
    """
    Black-Scholes model for European call option pricing and Greeks.
    """

    def __init__(self, S, K, T, r, sigma):
        self.S = float(S)
        self.K = float(K)
        self.T = float(T)
        self.r = float(r)
        self.sigma = float(sigma)

    def d1(self):
        return (
            np.log(self.S / self.K)
            + (self.r + 0.5 * self.sigma ** 2) * self.T
        ) / (self.sigma * np.sqrt(self.T))

    def d2(self):
        return self.d1() - self.sigma * np.sqrt(self.T)

    def call_price(self):
        d1 = self.d1()
        d2 = self.d2()

        return (
            self.S * norm.cdf(d1)
            - self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
        )

    def delta(self):
        return norm.cdf(self.d1())

    def gamma(self):
        return norm.pdf(self.d1()) / (self.S * self.sigma * np.sqrt(self.T))

    def vega(self):
        # Per 1% change in volatility
        return self.S * norm.pdf(self.d1()) * np.sqrt(self.T) / 100

    def theta(self):
        # Per calendar day
        d1 = self.d1()
        d2 = self.d2()

        theta_annual = (
            -self.S * norm.pdf(d1) * self.sigma / (2 * np.sqrt(self.T))
            - self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
        )

        return theta_annual / 365

    def rho(self):
        # Per 1% change in interest rate
        d2 = self.d2()

        return self.K * self.T * np.exp(-self.r * self.T) * norm.cdf(d2) / 100

    def summary(self):
        result = {
            "Option Price": self.call_price(),
            "Delta": self.delta(),
            "Gamma": self.gamma(),
            "Vega": self.vega(),
            "Theta": self.theta(),
            "Rho": self.rho()
        }

        return pd.DataFrame(result, index=["European Call"]).T

Overwriting /content/drive/MyDrive/market_risk_project/src/black_scholes.py


In [27]:
import black_scholes
from black_scholes import BlackScholesOption

print("BlackScholesOption reloaded successfully.")

PROJECT_DIR = "/content/drive/MyDrive/market_risk_project"
SRC_DIR = f"{PROJECT_DIR}/src"
DATA_DIR = f"{PROJECT_DIR}/data"

sys.path.append(SRC_DIR)

print("Project directory:", PROJECT_DIR)
print("Source directory:", SRC_DIR)
print("Data directory:", DATA_DIR)

BlackScholesOption reloaded successfully.
Project directory: /content/drive/MyDrive/market_risk_project
Source directory: /content/drive/MyDrive/market_risk_project/src
Data directory: /content/drive/MyDrive/market_risk_project/data


In [18]:
prices = pd.read_csv(f"{DATA_DIR}/prices.csv", index_col=0, parse_dates=True)
returns = pd.read_csv(f"{DATA_DIR}/returns.csv", index_col=0, parse_dates=True)

prices.tail()

,BND,GLD,QQQ,SPY,VTI
Date,,,,,
2026-06-24,73.305534,365.920013,710.619995,733.239990,362.606934
2026-06-25,73.355370,369.459991,716.380005,734.299988,362.936005
2026-06-26,73.425133,373.630005,706.520020,728.989990,362.220001
2026-06-29,73.465004,368.579987,724.080017,741.000000,367.119995
2026-06-30,73.166000,368.380005,736.400024,746.770020,370.040009


# Define option input:



In [19]:
# Current SPY price
S = prices["SPY"].iloc[-1]

# Strike price
K = S

# Time to maturity
# 3 months = 0.25 years
T = 0.25

# Risk-free rate
# Assumption: 4% annual risk-free rate
r = 0.04

# Annualized volatility of SPY
# Daily volatility × sqrt(252)
sigma = returns["SPY"].std() * np.sqrt(252)

print("SPY current price:", round(S, 2))
print("Strike price:", round(K, 2))
print("Time to maturity:", T)
print("Risk-free rate:", r)
print("Annualized volatility:", round(sigma, 4))

SPY current price: 746.77
Strike price: 746.77
Time to maturity: 0.25
Risk-free rate: 0.04
Annualized volatility: 0.1769


In [26]:

print("BlackScholesOption reloaded successfully.")
spy_call = BlackScholesOption(
    S=S,
    K=K,
    T=T,
    r=r,
    sigma=sigma
)

option_summary = spy_call.summary()
option_summary

BlackScholesOption reloaded successfully.


,European Call
Option Price,30.093413
Delta,0.562489
Gamma,0.005966
Vega,1.471279
Theta,-0.185347
Rho,0.974892


In [28]:
option_summary.to_csv(f"{DATA_DIR}/black_scholes_greeks.csv")

print("Saved to:")
print(f"{DATA_DIR}/black_scholes_greeks.csv")

Saved to:
/content/drive/MyDrive/market_risk_project/data/black_scholes_greeks.csv


In [29]:
option_summary_display = option_summary.copy()
option_summary_display["European Call"] = option_summary_display["European Call"].round(6)

option_summary_display

,European Call
Option Price,30.093413
Delta,0.562489
Gamma,0.005966
Vega,1.471279
Theta,-0.185347
Rho,0.974892


# Add sensativity shock
1. SPY Price Shock

In [30]:
# Create SPY price shock scenarios
price_shocks = [-0.10, -0.05, 0.00, 0.05, 0.10]

price_sensitivity_results = []

for shock in price_shocks:
    shocked_S = S * (1 + shock)

    option = BlackScholesOption(
        S=shocked_S,
        K=K,
        T=T,
        r=r,
        sigma=sigma
    )

    price_sensitivity_results.append({
        "SPY Price Shock": shock,
        "Shocked SPY Price": shocked_S,
        "Option Price": option.call_price(),
        "Delta": option.delta(),
        "Gamma": option.gamma(),
        "Vega": option.vega(),
        "Theta": option.theta(),
        "Rho": option.rho()
    })

price_sensitivity_df = pd.DataFrame(price_sensitivity_results)

price_sensitivity_df

,SPY Price Shock,Shocked SPY Price,Option Price,Delta,Gamma,Vega,Theta,Rho
0,-0.10,672.093018,4.463694,0.150588,0.003932,0.785566,-0.086748,0.241863
1,-0.05,709.431519,13.331290,0.336281,0.005815,1.294207,-0.150132,0.563092
2,0.00,746.770020,30.093413,0.562489,0.005966,1.471279,-0.185347,0.974892
3,0.05,784.108521,54.974396,0.760808,0.004474,1.216551,-0.177272,1.353953
4,0.10,821.447021,86.046914,0.891558,0.002562,0.764432,-0.144926,1.615802


In [31]:
price_sensitivity_df.to_csv(
    f"{DATA_DIR}/option_price_sensitivity.csv",
    index=False
)

print("Saved to:")
print(f"{DATA_DIR}/option_price_sensitivity.csv")

Saved to:
/content/drive/MyDrive/market_risk_project/data/option_price_sensitivity.csv


2. Add Volatility Sensitivity

In [34]:
vol_shocks = [-0.20, -0.10, 0.00, 0.10, 0.20]

vol_sensitivity_results = []

for shock in vol_shocks:
    shocked_sigma = sigma * (1 + shock)

    option = BlackScholesOption(
        S=S,
        K=K,
        T=T,
        r=r,
        sigma=shocked_sigma
    )

    vol_sensitivity_results.append({
        "Volatility Shock": shock,
        "Shocked Volatility": shocked_sigma,
        "Option Price": option.call_price(),
        "Delta": option.delta(),
        "Gamma": option.gamma(),
        "Vega": option.vega(),
        "Theta": option.theta(),
        "Rho": option.rho()
    })

vol_sensitivity_df = pd.DataFrame(vol_sensitivity_results)

vol_sensitivity_df

,Volatility Shock,Shocked Volatility,Option Price,Delta,Gamma,Vega,Theta,Rho
0,-0.2,0.141519,24.895442,0.570130,0.007433,1.466516,-0.157650,1.002150
1,-0.1,0.159208,27.492349,0.565695,0.006620,1.469348,-0.171465,0.987379
2,0.0,0.176898,30.093413,0.562489,0.005966,1.471279,-0.185347,0.974892
3,0.1,0.194588,32.697328,0.560181,0.005428,1.472610,-0.199276,0.964073
4,0.2,0.212278,35.303205,0.558547,0.004979,1.473521,-0.213236,0.954507


In [35]:
vol_sensitivity_df.to_csv(
    f"{DATA_DIR}/option_volatility_sensitivity.csv",
    index=False
)

print("Saved to:")
print(f"{DATA_DIR}/option_volatility_sensitivity.csv")

Saved to:
/content/drive/MyDrive/market_risk_project/data/option_volatility_sensitivity.csv
